### Implementation of the Indexing pipeline

Step 1: Data loading is performed using Langchain_community package but importing PyPDFLoader. loader object is created to the PyPDFLoader and using loader.load() method the data is loaded into a list

In [16]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!pip install langchain_community
%pip install langchain-openai==0.3.7
%pip install --upgrade chromadb hnswlib
#%pip install chromadb==0.4.10
# %pip install --upgrade langchain langchain-community langchain-openai openai tiktoken gradio pypdf "chromadb==0.4.16" "huggingface_hub==0.19.4"

  Using cached langchain_core-1.2.17-py3-none-any.whl.metadata (4.4 kB)
Using cached langchain_core-1.2.17-py3-none-any.whl (502 kB)
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 0.3.83
    Uninstalling langchain-core-0.3.83:
      Successfully uninstalled langchain-core-0.3.83
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain-openai 0.3.7 requires langchain-core<1.0.0,>=0.3.39, but you have langchain-core 1.2.17 which is incompatible.


  Using cached langchain_core-0.3.83-py3-none-any.whl.metadata (3.2 kB)
Using cached langchain_core-0.3.83-py3-none-any.whl (458 kB)
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/base_command.py", line 179, in exc_logging_wrapper
    status = run_func(*args)
             ^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/req_command.py", line 67, in wrapper
    return func(self, options, args)
           ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/commands/install.py", line 447, in run
    conflicts = self._determine_conflicts(to_install)
                ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/commands/install.py", line 578, in _determine_conflicts
    return check_install_conflicts(to_install)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_intern

In [1]:
from langchain_community.document_loaders import PyPDFLoader

loader=PyPDFLoader('/content/drive/MyDrive/Deep Learning IITG/Project/Course End Gen AI Project/the_nestle_hr_policy_pdf_2012.pdf')

In [2]:
pages = loader.load()

Step 2: Data chunking involved 3 steps

1. Splitting the data into smaller context windows
2. Merging fixed number of texts into smaller chunks
3. Create a new chunk with continuity from the previous chunk

There are various ways of performing data chunking or data splitting

1.Fixed length splitting, can be done on character level or token level with overlap between the chunks to maintain continuity. Preferred with large sized documents where the text is split based on the context window length or sequence length

2.Specialized or Adaptive length splitting, can be done in cases where the data structure is to be maintained like HTML data or in coding languages.

3.Semantic splitting, performed when the semantic relationship in the text is to be maintained like patient records or customer reviews.

In [3]:
###Fixed length splitting
from langchain_text_splitters import RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=150)
chunks=text_splitter.split_documents(pages)

Step 3: Embeddings

Since the raw data needs to converted into numerical representation we need to Embed the data. As per the requirement, using OpenAI's embedding model

In [4]:
from langchain_openai import OpenAIEmbeddings

In [5]:
import os
from getpass import getpass

os.environ["OPENAI_API_KEY"] = getpass('Enter your OpenAI key:')



Enter your OpenAI key:··········


In [6]:
#from langchain_openai import OpenAIEmbeddings
embeddings=OpenAIEmbeddings(model='text-embedding-3-small')

Step 4: Create vector representations for the document and store it in ChromaDB. The embeddings OPENAI object is passed into chroma.from_documents method to create vector representations using the embedding function like OpenAI used above.

In [7]:
from langchain_community.vectorstores import Chroma
vector_store=Chroma.from_documents(documents=chunks,
                                  embedding=embeddings,
                                  persist_directory='/content/drive/MyDrive/Deep Learning IITG/Project/RAG/ChromaDB')
vector_store.persist()

/tmp/ipykernel_41349/2452834701.py:5: LangChainDeprecationWarning: Since Chroma 0.4.x the manual persistence method is no longer supported as docs are automatically persisted.
  vector_store.persist()


In [8]:
## Fetching the vectors
vector_store=Chroma(persist_directory='/content/drive/MyDrive/Deep Learning IITG/Project/RAG/ChromaDB',
                    embedding_function=embeddings)

/tmp/ipykernel_41349/3914371512.py:2: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vector_store=Chroma(persist_directory='/content/drive/MyDrive/Deep Learning IITG/Project/RAG/ChromaDB',


### Implementation of the Generation pipeline

The Indexing pipeline loads the documents or data from external source there by creating a non parameteric memory for the model.

Generation pipeline involves 3 major steps

1. Retrieval, helps with the retrieval of the vector embedding matching the user prompt using various techniques like similarty search based on cosine similarity or Euclidean distance among many others. Retrieval directly affects the model output, the more relevant the data is retrieved the more accurate and relevant the output from the LLM

2. Agumentation, this process involves agumenting the user prompt(by converting those into embedding) with the retrival output and this is provided to LLM to generate the output.This process involves prompt engineering techniques to guide the LLM to generate more relevant and accurate texts

3. Generation, this step involves the response generated by the LLM

In [9]:
##Step 1: Retrieval
query = 'what is the long-term success of the company'

docs=vector_store.similarity_search(query,k=2)

In [10]:
print(docs[0].page_content)

The long-term success of the Company 
depends on its capacity to attract, retain and 
develop employees able to ensure ongoing 
and sustainable growth. This is a primary 
responsibility of all managers.
The Nestlé policy is to hire employees with 
personal attitudes and professional skills enabling 
them to develop a long-term relationship with the 
Company. Therefore, special attention will be paid 
to ensure there is a strong alignment between a 
candidate’s values and the Nestlé culture.
Only relevant skills and experience and 
adherence to the Nestlé principles will 
be considered in employing a person. No 
consideration will be given to a candidate’s origin, 
nationality, religion, race, gender, disability, sexual 
orientation or age.
Whilst adequate recruitment tools may


In [11]:
## Step2: Agumentation
## creating a prompt

context = '\n\n'.join([doc.page_content for doc in docs])

agumented_prompt = f"""

Given the below context, answer the question

Question: {query}

Context: {context}

Remember to answer only based on the context provided and not from any other source.

If the question cannot be answered based on the provided context, say I don't know.

"""

In [12]:
### Step3: Generation
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage

llm = ChatOpenAI(model='gpt-3.5-turbo')
response = llm.invoke([HumanMessage(content=agumented_prompt)])
answer = response.content
print(answer)

Based on the context provided, the long-term success of the company depends on its capacity to attract, retain, and develop employees who can ensure ongoing and sustainable growth.


In [14]:
import gradio
def generate_answer(query):
  docs=vector_store.similarity_search(query,k=2)
  context='\n\n'.join([doc.page_content for doc in docs])
  prompt = f"""
  Given the below context, answer the question

  Question: {query}

  Context: {context}

  Remember to answer only based on the context provided and not from any other source.

  If the question cannot be answered based on the provided context, say I don't know.

  """
  response = llm.invoke([HumanMessage(content=prompt)])
  return response.content

demo=gradio.Interface(fn=generate_answer,
                      inputs=gradio.Textbox(label='As a question about Nestle HR policy'),
                      outputs=gradio.Textbox(label='Answer'))

demo.launch()

Setting queue=True in a Colab notebook requires sharing enabled. Setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
Running on public URL: https://ade59565f76d008ddb.gradio.live

This share link expires in 72 hours. For free permanent hosting and GPU upgrades, run `gradio deploy` from Terminal to deploy to Spaces (https://huggingface.co/spaces)
